In [18]:
# # sanity_self_retrieval.ipynb
from __future__ import annotations
import json, random
from pathlib import Path
from importlib import import_module
import numpy as np
import pandas as pd
from scipy import sparse

INDEX_ROOT = Path("/work/index")
DATA_DIR   = Path("/work/data/processed")   # RAW jsons viven aquí
COLLECTION = "OEB"                          # cambia si procede

RAW_TEXTO_JSON = DATA_DIR / f"{COLLECTION}_texto.json"   # colección larga RAW (json)

# ---------- helpers ----------
def _read_json(p: Path) -> dict:
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def read_registry(index_root: Path) -> dict:
    p = index_root / "registry.json"
    return _read_json(p) if p.exists() else {}

def resolve_index_dir(method_name: str, index_root: Path) -> Path:
    reg = read_registry(index_root)
    if method_name in reg and reg[method_name].get("path"):
        return Path(reg[method_name]["path"])
    return index_root / method_name

def load_raw_texto_df(path: Path) -> pd.DataFrame:
    obj = _read_json(path)
    if isinstance(obj, list):
        docs = obj
    elif isinstance(obj, dict):
        for key in ("documents","data","items","records"):
            if key in obj and isinstance(obj[key], list):
                docs = obj[key]; break
        else:
            raise TypeError(f"JSON no soportado en {path}")
    else:
        raise TypeError(f"Raíz JSON no soportada: {type(obj)}")

    rows = []
    for d in docs:
        md = d.get("metadata", {}) if isinstance(d, dict) else {}
        rows.append({
            "item_key": (md.get("item_key") or d.get("item_key") or "").strip(),
            "text": d.get("text") or d.get("content") or md.get("text") or "",
        })
    df = pd.DataFrame(rows)
    df = df.replace({np.nan: ""})
    df = df[(df["item_key"] != "") & (df["text"].str.strip() != "")]
    return df.reset_index(drop=True)

def dynamic_load_retriever(method_name: str):
    """
    Importa el módulo retriever desde /work/src/retrievers/<method>.py
    Debe exponer: load(index_dir) -> searcher con:
      - search(query, k) y/o search_batch(list, k)
      - external_ids (np.array de item_keys)
    Para léxicos: además expose:
      * TF-IDF: .encode() y .X_docs  (cosine):contentReference[oaicite:2]{index=2}
      * BM25  : ._encode_queries_idf() y .X_docs  (producto Q_idf @ X_docs^T):contentReference[oaicite:3]{index=3}
    """
    import sys
    src = "/work/src"
    if src not in sys.path:
        sys.path.insert(0, src)
    return import_module(f"retrievers.{method_name}")

def _encode_query_matrix(searcher, text: str):
    """
    Devuelve matriz 1xV del query si el retriever lo permite.
    - TF-IDF → encode([text])
    - BM25   → _encode_queries_idf([text])
    Densos: retornamos None (se manejará por fallback).
    """
    if hasattr(searcher, "encode"):               # TF-IDF
        return searcher.encode([text])
    if hasattr(searcher, "_encode_queries_idf"):  # BM25
        return searcher._encode_queries_idf([text])
    return None

def _get_doc_vector(searcher, internal_id: int):
    """
    Intenta obtener el vector del documento para score directo:
      - .X_docs[i] (TF-IDF/BM25)
      - .doc_matrix[i], .embeddings[i], etc. (densos)
      - .get_doc_vector(internal_id) si existe
    Si no se puede, devuelve None.
    """
    for attr in ("X_docs","doc_matrix","embeddings","D","docs"):
        if hasattr(searcher, attr):
            M = getattr(searcher, attr)
            try:
                return M[internal_id]
            except Exception:
                pass
    if hasattr(searcher, "get_doc_vector"):
        try:
            return searcher.get_doc_vector(internal_id)
        except Exception:
            pass
    return None

def _score_query_doc(Q, drow):
    """
    Calcula score(Q, d) de forma robusta:
      - Si Q es CSR/array y drow es fila CSR → (Q @ d.T)
      - Si ambos son arrays densos → np.dot
      - Si tipos no compatibles → None
    """
    if Q is None or drow is None:
        return None
    # CSR @ CSR^T
    if sparse.isspmatrix_csr(Q) and sparse.isspmatrix(drow):
        return float((Q @ drow.T).toarray().ravel()[0])
    # mixtos: convertir a densos pequeños
    try:
        qv = Q.toarray().ravel() if hasattr(Q, "toarray") else np.asarray(Q).ravel()
        dv = drow.toarray().ravel() if hasattr(drow, "toarray") else np.asarray(drow).ravel()
        return float(np.dot(qv, dv))
    except Exception:
        return None

def sanity_self_retrieval(method_name: str, k: int = 5, seed: int = 0):
    # 1) sample aleatorio de RAW texto
    df_long_raw = load_raw_texto_df(RAW_TEXTO_JSON)
    if df_long_raw.empty:
        raise RuntimeError(f"Sin filas válidas en {RAW_TEXTO_JSON}")
    rnd = random.Random(seed)
    row = df_long_raw.iloc[rnd.randrange(len(df_long_raw))]
    gold_key = str(row["item_key"])
    query_txt = str(row["text"])

    # 2) resolver índice
    index_dir = resolve_index_dir(method_name, INDEX_ROOT)
    if not index_dir.exists():
        raise FileNotFoundError(f"No existe índice para {method_name}: {index_dir}")

    # 3) cargar retriever y buscar
    retr_mod = dynamic_load_retriever(method_name)
    searcher = retr_mod.load(index_dir)  # contrato común

    top_idx, top_s = searcher.search(query_txt, k=max(k, 10))
    ext_ids = np.asarray(searcher.external_ids, dtype=object)
    if top_idx.size == 0:
        raise RuntimeError("Búsqueda devolvió 0 candidatos")

    top1_internal = int(top_idx[0])
    top1_key = str(ext_ids[top1_internal])
    top1_score = float(top_s[0])

    # 4) score del GOLD
    #   (a) intento directo con encode()+doc_vector
    Q = _encode_query_matrix(searcher, query_txt)
    gold_internal = None
    try:
        # localizar el doc interno del gold (O(N) una sola vez)
        gold_internal = int(np.where(ext_ids == gold_key)[0][0])
    except Exception:
        pass

    gold_score = None
    if gold_internal is not None:
        drow = _get_doc_vector(searcher, gold_internal)
        gold_score = _score_query_doc(Q, drow)

    #   (b) si no hay forma directa, buscamos a N completo (1 query → coste asumible)
    if gold_score is None:
        try:
            top_idx_full, top_s_full = searcher.search(query_txt, k=len(ext_ids))
            # ordenar ya viene por score desc según contrato de search()
            mask = (ext_ids[top_idx_full] == gold_key)
            if mask.any():
                gold_score = float(top_s_full[np.where(mask)[0][0]])
        except Exception:
            gold_score = None  # último recurso: reportar None

    # 5) salida
    is_top1_gold = (top1_key == gold_key)
    return {
        "method": method_name,
        "sampled_item_key": gold_key,
        "text_len": len(query_txt),
        "top1_item_key": top1_key,
        "top1_score": top1_score,
        "is_top1_gold": bool(is_top1_gold),
        "gold_score": gold_score,
    }

def run_and_print(method_name: str, k: int = 5, seed: int = 0):
    r = sanity_self_retrieval(method_name, k=k, seed=seed)
    print(f"[{r['method']}] sample={r['sampled_item_key']} len={r['text_len']}")
    print(f"  top1:  {r['top1_item_key']}  score={r['top1_score']:.6f}  (is_gold={r['is_top1_gold']})")
    if r["gold_score"] is not None:
        print(f"  gold:  {r['sampled_item_key']}  score={r['gold_score']:.6f}")
    else:
        print("  gold:  <no se pudo calcular directamente; se necesita API de vectores o k=N>")

# Ejemplos:
# run_and_print("tfidf_unigram")
# run_and_print("bm25_unigram")
# run_and_print("tfidf_char_3_5")
# run_and_print("dense_e5") 
# run_and_print("dense_gte")        # o el nombre que uses para e5/gte/colbert/splade


AttributeError: 'Searcher' object has no attribute 'search'